# Legal GPT evaluation on Kaggle

Runs both Legal tab tests on Kaggle GPUs:

* **Test 1** -- the 16-question elections-law benchmark (`evals/legal/elections_2026_he.json`) over an index of
  that one law, so results compare with the earlier local runs.
* **Test 2** -- the 24-question multi-law test (`evals/legal/multi_law_24_questions.json`) over an index of all
  five laws in `legal_txt/`. The gold file is read only by the grading phase, never by the model under test.

Answers come from `qwen3:8b` (as locally); grading uses `qwen3:14b`. Both run in Ollama on the GPUs.

**Before running** (notebook panel on the right):
1. *Settings -> Accelerator*: **GPU T4 x2**. *Settings -> Internet*: **On**.
2. *Add-ons -> Secrets*: add and tick **GITHUB_TOKEN** (a GitHub token that can read `zananiri/AI-IZ`) and
   **DOCSLIDES_LEGAL_BUNDLE_KEY** (any long random string -- it only signs the index built here).
3. Set `BRANCH` below to the branch with the code to test.
4. *Save Version -> Save & Run All (Commit)*. It runs in the background (about 2-4 hours);
   download `legal_eval_results.zip` from the version's *Output* tab when it finishes.

In [ ]:
BRANCH = "main"            # the branch to test
ANSWER_MODEL = "qwen3:8b"  # the model under test (same as the local runs)
JUDGE_MODEL = "qwen3:14b"  # the grader

## 1. Code and Python packages

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
token = secrets.get_secret("GITHUB_TOKEN")
os.environ["DOCSLIDES_LEGAL_BUNDLE_KEY"] = secrets.get_secret("DOCSLIDES_LEGAL_BUNDLE_KEY")

REPO = "/kaggle/working/AI-IZ"
!git clone -q --depth 1 -b {BRANCH} https://{token}@github.com/zananiri/AI-IZ.git {REPO}
!git -C {REPO} remote set-url origin https://github.com/zananiri/AI-IZ.git
%cd {REPO}
!git log --oneline -1
!pip install -q -e ".[legal,dev]"

## 2. Ollama with the answer and judge models

In [ ]:
!apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, urllib.request
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"),
                          stderr=subprocess.STDOUT, env={**os.environ, "OLLAMA_MAX_LOADED_MODELS": "2"})
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        break
    except Exception:
        time.sleep(2)
!ollama pull {ANSWER_MODEL}
!ollama pull {JUDGE_MODEL}
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Point the app at Ollama, and make one config per index

In [ ]:
import copy, pathlib, shutil, yaml

os.environ.update({
    "DOCSLIDES_LLM_BACKEND": "ollama", "DOCSLIDES_LLM_BASE_URL": "http://localhost:11434",
    "DOCSLIDES_LLM_MODEL": ANSWER_MODEL,
    "DOCSLIDES_LEGAL_ORCHESTRATOR_BACKEND": "ollama", "DOCSLIDES_LEGAL_ORCHESTRATOR_BASE_URL": "http://localhost:11434",
    "DOCSLIDES_LEGAL_ORCHESTRATOR_MODEL": ANSWER_MODEL,
    "DOCSLIDES_LEGAL_JUDGE_MODEL": JUDGE_MODEL,
})

base = yaml.safe_load(open("config/config.yaml", encoding="utf-8"))

def make_config(name, laws_dir):
    cfg = copy.deepcopy(base)
    legal = cfg["legal"]
    legal["retrieval"]["vectordb_dir"] = f"./data/{name}/vectordb"
    legal["ingestion"]["legal_txt_dir"] = laws_dir
    legal["ingestion"]["staging_dir"] = f"./data/{name}/staging"
    legal["ingestion"]["bundle_manifest"] = f"./data/{name}/signed_bundle.json"
    legal["audit_dir"] = f"./data/{name}/audit"
    for sub in ("staging", "audit"):
        pathlib.Path(f"data/{name}/{sub}").mkdir(parents=True, exist_ok=True)
    path = f"config/kaggle_{name}.yaml"
    yaml.safe_dump(cfg, open(path, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)
    return path

# Test 1 indexes only the elections law, like the earlier local runs.
single = pathlib.Path("legal_txt_single"); single.mkdir(exist_ok=True)
for name in ("25_lsr_14147458.pdf", "25_lsr_14147458.pdf.meta.json"):
    shutil.copy(f"legal_txt/{name}", single / name)
CONFIG_SINGLE = make_config("single", "./legal_txt_single")
CONFIG_MULTI = make_config("multi", "./legal_txt")
pathlib.Path("data/legal/eval").mkdir(parents=True, exist_ok=True)

Quick check that the code works here (about a minute).

In [ ]:
!python -m pytest tests/unit -q -p no:cacheprovider 2>&1 | tail -3

## 4. Test 1 -- 16 questions, elections law only

In [ ]:
os.environ["DOCSLIDES_CONFIG"] = CONFIG_SINGLE
!python scripts/ingest_legal_txt.py
!python scripts/eval_retrieval.py --json data/legal/eval/kaggle_retrieval16.json
!python scripts/eval_legal.py --run-dir data/legal/eval/kaggle_run16

## 5. Test 2 -- 24 questions, five laws

In [ ]:
os.environ["DOCSLIDES_CONFIG"] = CONFIG_MULTI
!python scripts/ingest_legal_txt.py | tee data/legal/eval/kaggle_multi24_ingest.txt
!python scripts/eval_legal_gold.py --retrieval-only | tee data/legal/eval/kaggle_multi24_retrieval.txt
!python scripts/eval_legal_gold.py --run-dir data/legal/eval/kaggle_multi24

## 6. Results to download

In [ ]:
!cd /kaggle/working && zip -qr legal_eval_results.zip AI-IZ/data/legal/eval AI-IZ/data/single/audit AI-IZ/data/multi/audit ollama.log
!ls -lh /kaggle/working/legal_eval_results.zip
!cat data/legal/eval/kaggle_run16/summary.json
!cat data/legal/eval/kaggle_multi24/summary.json